# BINN-style sequential MPNN on the layered Reactome graph

This notebook replaces shallow synchronous GCN/GAT layers on the **raw** Reactome DAG with a **layer-by-layer message passing model** on the **strictly layered BINN graph** created in notebook 1.

## Why this change matters

Your graph preparation notebook already creates `outputs/graph_layered_binn/`, where every edge goes from layer `ℓ -> ℓ+1`.  
That is the right structure for a biological bottom-up hierarchy.

The original training notebook never used that artifact. Instead, it trained shallow PyG `GCNConv/GATConv` models on the raw DAG. That creates a mismatch:

- expression exists only on gene nodes
- roots are many hops away
- shallow synchronous convolutions only expose a limited receptive field
- gene pooling is misaligned with the edge direction
- all-node pooling dilutes the signal with thousands of mostly uninitialized pathway nodes

The fix below uses:

1. `gene_layered_ids_in_expr_order.pt` to inject expression at layer-0 genes
2. `layer_edges.pt` to propagate one biological hop at a time
3. `root_pathways_layered_ids` as the only readout nodes
4. unit tests and diagnostics to verify that signal and gradients really reach the top

In [ ]:
# === 0) Paths, imports, and runtime ===
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

BASE_DIR = Path(os.environ.get("BINN_GNN_BASE", ".")).resolve()
OUT_DIR = BASE_DIR / "outputs"

LAYERED_GRAPH_DIR = OUT_DIR / "graph_layered_binn"
EXPR_PARQUET = OUT_DIR / "expr_reactome_tcga_tumor_normal.parquet"
Y_CSV = OUT_DIR / "y_tcga_tumor_normal.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("BASE_DIR:", BASE_DIR)
print("device:", device)

required = [
    EXPR_PARQUET,
    Y_CSV,
    LAYERED_GRAPH_DIR / "node_table_layered.csv",
    LAYERED_GRAPH_DIR / "edge_index_layered.pt",
    LAYERED_GRAPH_DIR / "layer_info.json",
    LAYERED_GRAPH_DIR / "gene_layered_ids_in_expr_order.pt",
]
missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Missing required artifacts. Run the adjusted graph-preparation notebook first.\n"
        + "\n".join(missing)
    )

In [ ]:
# === 1) Load expression, labels, and layered graph artifacts ===
expr = pd.read_parquet(EXPR_PARQUET)   # genes x samples
y_df = pd.read_csv(Y_CSV, index_col=0)
samples = y_df.index.astype(str).tolist()
y = y_df.iloc[:, 0].to_numpy(dtype=np.int64)

expr = expr.loc[:, samples]

node_table = pd.read_csv(LAYERED_GRAPH_DIR / "node_table_layered.csv")
edge_index = torch.load(LAYERED_GRAPH_DIR / "edge_index_layered.pt").long()
gene_layered_ids_in_expr_order = torch.load(
    LAYERED_GRAPH_DIR / "gene_layered_ids_in_expr_order.pt"
).long()

with open(LAYERED_GRAPH_DIR / "layer_info.json", "r") as f:
    layer_info = json.load(f)

node_id_col = "layered_id" if "layered_id" in node_table.columns else "node_id"
name_col = "orig_name" if "orig_name" in node_table.columns else "node_name"

num_nodes = int(node_table[node_id_col].max()) + 1
node_layers = (
    torch.load(LAYERED_GRAPH_DIR / "node_layers.pt").long()
    if (LAYERED_GRAPH_DIR / "node_layers.pt").exists()
    else torch.tensor(node_table.sort_values(node_id_col)["layer"].to_numpy(), dtype=torch.long)
)

if (LAYERED_GRAPH_DIR / "layer_edges.pt").exists():
    layer_edges = [e.long() for e in torch.load(LAYERED_GRAPH_DIR / "layer_edges.pt")]
else:
    src_layers = node_layers[edge_index[0]]
    dst_layers = node_layers[edge_index[1]]
    assert torch.all(dst_layers == src_layers + 1), "layered edge_index must satisfy layer+1 edges"
    layer_edges = [edge_index[:, src_layers == l].clone() for l in range(int(node_layers.max().item()))]

root_ids = (
    torch.load(LAYERED_GRAPH_DIR / "root_layered_ids.pt").long()
    if (LAYERED_GRAPH_DIR / "root_layered_ids.pt").exists()
    else torch.tensor(layer_info["root_pathways_layered_ids"], dtype=torch.long)
)

print("expr:", expr.shape, "(genes x samples)")
print("labels:", y.shape, "pos rate:", float(y.mean()))
print("layered num_nodes:", num_nodes, "| num_edges:", edge_index.shape[1])
print("num_layers:", len(layer_edges), "| max layer:", int(node_layers.max().item()))
print("num root readout nodes:", int(root_ids.numel()))
print("gene nodes used for injection:", int(gene_layered_ids_in_expr_order.numel()))

In [ ]:
# === 2) Structural diagnostics: verify the training graph really matches the model ===
src_layers = node_layers[edge_index[0]]
dst_layers = node_layers[edge_index[1]]

G_dir = nx.DiGraph()
G_dir.add_nodes_from(range(num_nodes))
G_dir.add_edges_from(edge_index.t().tolist())

print("Is layered graph a DAG?", nx.is_directed_acyclic_graph(G_dir))
print("Every edge advances exactly one layer?", bool(torch.all(dst_layers == src_layers + 1)))
print("Layer sizes:")
display(node_table["layer"].value_counts().sort_index())

ordered_gene_names = (
    node_table
    .set_index(node_id_col)
    .loc[gene_layered_ids_in_expr_order.tolist(), name_col]
    .astype(str)
    .tolist()
)
print("Expression row order matches layered gene ids?", ordered_gene_names == expr.index.astype(str).tolist())
print("Roots at final layer only?", bool(torch.all(node_layers[root_ids] == node_layers.max())))

In [ ]:
# === 3) Build train/val arrays and loaders ===
X_all = expr.T.to_numpy(dtype=np.float32)   # samples x genes
assert X_all.shape[1] == len(gene_layered_ids_in_expr_order), "Mismatch between expr genes and saved graph gene ids"

train_idx, val_idx = train_test_split(
    np.arange(X_all.shape[0]),
    test_size=0.2,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_all[train_idx] = scaler.fit_transform(X_all[train_idx])
X_all[val_idx] = scaler.transform(X_all[val_idx])

# Keep this True for fast structural debugging; switch to False once the model behaves.
DEBUG_BALANCED_SUBSET = True
MAX_PER_CLASS = 512

rng = np.random.default_rng(0)
if DEBUG_BALANCED_SUBSET:
    def balanced_subset(indices):
        y_part = y[indices]
        pos = indices[y_part == 1]
        neg = indices[y_part == 0]
        n = min(len(pos), len(neg), MAX_PER_CLASS)
        return np.concatenate([
            rng.choice(pos, n, replace=False),
            rng.choice(neg, n, replace=False),
        ])
    train_idx = balanced_subset(train_idx)
    val_idx = balanced_subset(val_idx)

print("train:", len(train_idx), "val:", len(val_idx))
print("train pos rate:", float(y[train_idx].mean()), "| val pos rate:", float(y[val_idx].mean()))

class ExpressionDataset(Dataset):
    def __init__(self, X, y, indices):
        self.X = X
        self.y = y
        self.indices = np.array(indices, dtype=int)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        i = int(self.indices[k])
        x = torch.from_numpy(self.X[i]).float()
        label = torch.tensor(float(self.y[i]), dtype=torch.float32)
        return x, label

train_ds = ExpressionDataset(X_all, y, train_idx)
val_ds = ExpressionDataset(X_all, y, val_idx)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

In [ ]:
# === 4) Sequential BINN-style MPNN ===
def make_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == "tanh":
        return nn.Tanh()
    if name == "elu":
        return nn.ELU()
    if name == "relu":
        return nn.ReLU()
    if name == "identity":
        return nn.Identity()
    raise ValueError(f"Unknown activation: {name}")

class BINNSequentialMPNN(nn.Module):
    """
    Layer-by-layer message passing on a strictly layered DAG.

    - Input signal is injected only at layer-0 gene nodes.
    - Each edge has one learnable scalar weight.
    - Messages are aggregated with scatter_add_.
    - Readout uses only final-layer root pathways.
    """
    def __init__(
        self,
        num_nodes: int,
        layer_edges,
        gene_ids: torch.Tensor,
        root_ids: torch.Tensor,
        activation: str = "tanh",
        dropout: float = 0.0,
        edge_init_std: float = 0.02,
    ):
        super().__init__()
        self.num_nodes = int(num_nodes)
        self.num_layers = len(layer_edges)
        self.dropout = float(dropout)
        self.activation_name = activation
        self.activation = make_activation(activation)

        self.register_buffer("gene_ids", gene_ids.long())
        self.register_buffer("root_ids", root_ids.long())

        self.node_bias = nn.Parameter(torch.zeros(self.num_nodes))
        self.edge_weights = nn.ParameterList()

        for i, edges in enumerate(layer_edges):
            edges = edges.long()
            self.register_buffer(f"edge_index_{i}", edges)
            self.register_buffer(f"dst_ids_{i}", torch.unique(edges[1]).long())
            self.edge_weights.append(
                nn.Parameter(torch.randn(edges.shape[1]) * edge_init_std)
            )

        self.readout = nn.Linear(int(root_ids.numel()), 1)

    def forward(self, x_genes: torch.Tensor, return_debug: bool = False):
        batch_size = x_genes.size(0)

        # Initialize full node state with learnable pathway/node biases.
        h = self.node_bias.unsqueeze(0).expand(batch_size, -1).clone()

        # Inject observed expression only into layer-0 genes.
        h[:, self.gene_ids] = h[:, self.gene_ids] + x_genes

        debug = None
        if return_debug:
            debug = {"activations": [h[:, self.gene_ids].detach().cpu().reshape(-1)]}

        for layer_idx in range(self.num_layers):
            edges = getattr(self, f"edge_index_{layer_idx}")
            dst_ids = getattr(self, f"dst_ids_{layer_idx}")

            src = edges[0]
            dst = edges[1]

            msg = h.index_select(1, src) * self.edge_weights[layer_idx].view(1, -1)

            updates = h.new_zeros(batch_size, self.num_nodes)
            updates.scatter_add_(1, dst.view(1, -1).expand(batch_size, -1), msg)

            pre_act = updates.index_select(1, dst_ids) + self.node_bias.index_select(0, dst_ids).view(1, -1)
            h[:, dst_ids] = self.activation(pre_act)

            if self.dropout > 0:
                h[:, dst_ids] = F.dropout(h[:, dst_ids], p=self.dropout, training=self.training)

            if return_debug:
                debug["activations"].append(h[:, dst_ids].detach().cpu().reshape(-1))

        root_state = h.index_select(1, self.root_ids)
        logits = self.readout(root_state).view(-1)

        if return_debug:
            debug["root_state"] = root_state.detach().cpu()
            return logits, debug

        return logits

def eval_auc(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            prob = torch.sigmoid(logits).cpu().numpy()
            ys.append(yb.numpy())
            ps.append(prob)
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(ps)
    return float(roc_auc_score(y_true, y_pred))

def train_model(model, epochs=30, lr=5e-3, weight_decay=1e-6, use_pos_weight=False):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    y_train = y[train_idx]
    n_pos = int((y_train == 1).sum())
    n_neg = int((y_train == 0).sum())

    if use_pos_weight and n_pos > 0:
        pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        print("Using pos_weight:", float(pos_weight.item()))
    else:
        loss_fn = nn.BCEWithLogitsLoss()

    hist = {"train_loss": [], "val_auc": [], "grad_norms": []}

    for ep in range(1, epochs + 1):
        model.train()
        losses = []
        grad_accum = np.zeros(model.num_layers, dtype=np.float64)
        n_batches = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()

            grad_norms = np.array([
                0.0 if p.grad is None else float(p.grad.norm().item())
                for p in model.edge_weights
            ], dtype=np.float64)
            grad_accum += grad_norms
            n_batches += 1

            opt.step()
            losses.append(float(loss.item()))

        val_auc = eval_auc(model, val_loader)
        hist["train_loss"].append(float(np.mean(losses)))
        hist["val_auc"].append(val_auc)
        hist["grad_norms"].append((grad_accum / max(n_batches, 1)).tolist())

        print(
            f"epoch {ep:03d} | "
            f"loss {hist['train_loss'][-1]:.4f} | "
            f"val_auc {val_auc:.4f} | "
            f"grad(first,last)=({hist['grad_norms'][-1][0]:.3e}, {hist['grad_norms'][-1][-1]:.3e})"
        )

    return model, hist

def plot_activation_histograms(model, x_batch):
    model.eval()
    with torch.no_grad():
        _, debug = model(x_batch.to(device), return_debug=True)

    for layer_idx, vals in enumerate(debug["activations"]):
        vals = vals.numpy()
        plt.figure(figsize=(6, 3))
        plt.hist(vals, bins=50)
        plt.title(f"Activation histogram at hop/layer {layer_idx}")
        plt.xlabel("activation")
        plt.ylabel("count")
        plt.show()

def plot_gradient_norms(hist):
    grad_arr = np.array(hist["grad_norms"])
    plt.figure(figsize=(7, 4))
    for layer_idx in range(grad_arr.shape[1]):
        plt.plot(grad_arr[:, layer_idx], alpha=0.4)
    plt.title("Edge-weight gradient norms by layer across epochs")
    plt.xlabel("epoch")
    plt.ylabel("avg grad norm")
    plt.show()

In [ ]:
# === 5) Unit tests: structure and signal-flow smoke tests ===
def run_model_unit_tests():
    # 1) layer_edges must exactly partition the layered edge list
    reconstructed = torch.cat(layer_edges, dim=1)
    assert sorted(map(tuple, reconstructed.t().tolist())) == sorted(map(tuple, edge_index.t().tolist())),         "layer_edges does not match edge_index_layered"

    # 2) forward pass shape on the real graph
    model = BINNSequentialMPNN(
        num_nodes=num_nodes,
        layer_edges=layer_edges,
        gene_ids=gene_layered_ids_in_expr_order,
        root_ids=root_ids,
        activation="identity",
        dropout=0.0,
    ).to(device)

    xb = torch.randn(4, len(gene_layered_ids_in_expr_order), device=device)
    out = model(xb)
    assert out.shape == (4,), f"Expected shape (4,), got {tuple(out.shape)}"

    # 3) exact propagation on a toy 3-node chain: gene -> hidden -> root
    toy_layer_edges = [
        torch.tensor([[0], [1]], dtype=torch.long),
        torch.tensor([[1], [2]], dtype=torch.long),
    ]
    toy = BINNSequentialMPNN(
        num_nodes=3,
        layer_edges=toy_layer_edges,
        gene_ids=torch.tensor([0]),
        root_ids=torch.tensor([2]),
        activation="identity",
        dropout=0.0,
    ).to(device)

    with torch.no_grad():
        toy.node_bias.zero_()
        for w in toy.edge_weights:
            w.fill_(1.0)
        toy.readout.weight.fill_(1.0)
        toy.readout.bias.zero_()

    toy_out = toy(torch.tensor([[2.0]], dtype=torch.float32, device=device))
    assert torch.allclose(toy_out.detach().cpu(), torch.tensor([2.0]), atol=1e-6),         f"Toy chain should transmit the signal exactly, got {toy_out}"

    # 4) gradients must reach early layers on the real graph
    grad_model = BINNSequentialMPNN(
        num_nodes=num_nodes,
        layer_edges=layer_edges,
        gene_ids=gene_layered_ids_in_expr_order,
        root_ids=root_ids,
        activation="identity",
        dropout=0.0,
    ).to(device)

    xb = torch.from_numpy(X_all[train_idx[:8]]).to(device)
    yb = torch.tensor(y[train_idx[:8]], dtype=torch.float32, device=device)

    loss = nn.BCEWithLogitsLoss()(grad_model(xb), yb)
    loss.backward()

    grad_norms = [
        0.0 if p.grad is None else float(p.grad.norm().item())
        for p in grad_model.edge_weights
    ]
    assert any(g > 0 for g in grad_norms[: min(3, len(grad_norms))]),         "No gradient reached the earliest layers"

    print("All model unit tests passed.")

run_model_unit_tests()

In [ ]:
# === 6) Train the sequential model ===
EPOCHS = 30

model = BINNSequentialMPNN(
    num_nodes=num_nodes,
    layer_edges=layer_edges,
    gene_ids=gene_layered_ids_in_expr_order,
    root_ids=root_ids,
    activation="tanh",   # switch to "elu" if early-layer gradients still collapse
    dropout=0.0,
)

model, hist = train_model(
    model,
    epochs=EPOCHS,
    lr=5e-3,
    weight_decay=1e-6,
    use_pos_weight=not DEBUG_BALANCED_SUBSET,
)

best_auc = float(np.max(hist["val_auc"]))
best_epoch = int(np.argmax(hist["val_auc"])) + 1
print(f"Best val AUC: {best_auc:.4f} at epoch {best_epoch}")

plt.figure(figsize=(7, 4))
plt.plot(hist["train_loss"])
plt.title("Sequential BINN-MPNN train loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(hist["val_auc"])
plt.title("Sequential BINN-MPNN validation AUC")
plt.xlabel("epoch")
plt.ylabel("AUC")
plt.ylim(0, 1)
plt.show()

plot_gradient_norms(hist)

In [ ]:
# === 7) Activation diagnostics on one validation batch ===
xb_debug, yb_debug = next(iter(val_loader))
plot_activation_histograms(model, xb_debug[:32])

print("Tip:")
print("- If early-hop activations are almost all zero, lower edge_init_std or switch activation to ELU.")
print("- If early-layer gradient norms collapse to zero, add residual or skip connections from genes to roots.")
print("- Do not switch back to all-node pooling or raw shallow GCN/GAT until this sequential baseline is healthy.")

## What to compare against your old notebook

Use this notebook as the structural baseline.

A fair comparison is:

1. same train/val split
2. same standardized expression matrix
3. old raw-DAG GCN/GAT notebook
4. this sequential layered BINN-MPNN notebook

If the sequential model improves while the raw shallow models stay near chance, the bottleneck was the **message-passing schedule**, not the class weighting.